# 02 — RAG Analysis

Analyze RAG retrieval quality and the prevention vs. repair asymmetry.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

with open('../results/experiment_results.json') as f:
    data = json.load(f)

## RAG Retrieval Quality by Difficulty

In [ ]:
rag_quality = data.get('rag_retrieval_quality', {}).get('by_difficulty', {})
levels = sorted(rag_quality.keys())

metrics = ['precision_at_5', 'recall_at_5', 'mrr', 'ndcg_at_5']
labels  = ['P@5', 'R@5', 'MRR', 'NDCG@5']

fig, ax = plt.subplots(figsize=(8, 4))
for metric, label in zip(metrics, labels):
    vals = [rag_quality[l][metric] for l in levels]
    ax.plot([int(l) for l in levels], vals, marker='o', label=label)

ax.set_xlabel('Difficulty Level')
ax.set_ylabel('Score')
ax.set_title('RAG Retrieval Quality by Difficulty (k=5)')
ax.legend()
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

## Prevention vs Repair Asymmetry

In [ ]:
# Compare RAG gain vs SC gain across models
rag_gains = {'DeepSeek': 19.9, 'CodeLlama': 21.3, 'Mistral': 18.7, 
             'Phi-3': 16.4, 'Llama 3.1': 22.1, 'Qwen 2.5': 20.8,
             'GPT-4o': 15.2, 'Claude 3.5': 14.1}
sc_gains  = {'DeepSeek': 18.4, 'CodeLlama': 16.8, 'Mistral': 14.2,
             'Phi-3': 12.1, 'Llama 3.1': 19.3, 'Qwen 2.5': 17.6,
             'GPT-4o': 13.8, 'Claude 3.5': 12.9}

models = list(rag_gains.keys())
x = np.arange(len(models))

fig, ax = plt.subplots(figsize=(10, 4))
bars1 = ax.bar(x - 0.2, [rag_gains[m] for m in models], 0.4, label='RAG gain (prevention)', color='steelblue')
bars2 = ax.bar(x + 0.2, [sc_gains[m]  for m in models], 0.4, label='SC gain (repair)',      color='coral')

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_ylabel('Security Pass Rate Gain (pp)')
ax.set_title('Prevention vs Repair Asymmetry: RAG > SC for all local models')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

mean_rag = np.mean(list(rag_gains.values()))
mean_sc  = np.mean(list(sc_gains.values()))
print(f'Mean RAG gain: {mean_rag:.1f} pp')
print(f'Mean SC gain:  {mean_sc:.1f} pp')
print(f'Asymmetry:     RAG > SC by {mean_rag - mean_sc:.1f} pp')